<!-- cabecera-entorno -->
## Antes de empezar

**Clase 2 · Python y pandas: cargar, mirar, seleccionar y filtrar** — Bloque 3 · Reto. Este
cuaderno lo recorre **usted solo**, leyendo: cada tarea trae la explicación y los comandos que
necesita. El profesor circula por el salón resolviendo dudas. Es el entregable de la clase.

**La rutina de siempre:** `git pull` antes de clase, y el entorno virtual activo (`(.venv)` en la
terminal). Si va a modificar este archivo, trabaje sobre una copia: duplique `reto.ipynb` como
`reto_mio.ipynb` y edite el duplicado. Así `git pull` nunca le reclama.

**Si la celda de abajo falla, no siga:** la respuesta está en el manual del entorno,
[`../INSTALACION.md`](../INSTALACION.md).

| Si ve esto | Qué pasó | Dónde se arregla |
|------------|----------|------------------|
| `ModuleNotFoundError` | El entorno virtual no está activo, o VSCode eligió otro intérprete | Manual, secciones 6.3 y 8.4, y problema 5 |
| `FileNotFoundError` al leer el CSV | El cuaderno se abrió desde otra carpeta, o falta hacer `git pull` | Manual, problema 6 |
| El kernel no aparece en VSCode | Falta la extensión Jupyter o `ipykernel` dentro del entorno | Manual, problema 4 |

In [ ]:
# Verificación del entorno. Si algo falla aquí, la solución está en ../INSTALACION.md
import sys
from pathlib import Path

try:
    import pandas as pd
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        f"Falta la librería '{error.name}'. Active el entorno virtual y seleccione el intérprete "
        ".venv en VSCode (Ctrl+Shift+P > Python: Select Interpreter), luego reinicie el kernel. "
        "Ver ../INSTALACION.md, problema 5."
    ) from error

print("Intérprete:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este no parece el Python del entorno virtual. En VSCode: Ctrl+Shift+P >",
          "'Python: Select Interpreter' > el que dice .venv, y reinicie el kernel.")

RUTA_VERIFICACION = "../datos/vehiculos_accidentes.csv"
if Path(RUTA_VERIFICACION).exists():
    print("Datos: encontrados en", RUTA_VERIFICACION)
else:
    print("FALTA el archivo", RUTA_VERIFICACION, "- abra en VSCode la carpeta raíz del curso",
          "y ejecute 'git pull'. Ver ../INSTALACION.md, problema 6.")

# Clase 2 · Reto — Filtrar 20.000 accidentes de tránsito

**Dataset:** `../datos/vehiculos_accidentes.csv` (Ministerio de Transporte, datos.gov.co)
**Consigna completa:** `README.md`

20.000 filas. Una fila = un vehículo involucrado en un accidente de tránsito en Colombia entre
diciembre de 2022 y diciembre de 2025.

| Columna | Qué es |
|---------|--------|
| `marca_vehiculo` | Marca (AKT, YAMAHA, RENAULT...) |
| `modelo_vehiculo` | Año del modelo |
| `tipo_vehiculo` | MOTOCICLETA, AUTOMOVIL, CAMIONETA, BUS... |
| `edad_vehiculo` | Antigüedad en años |
| `fecha_accidente` | Mes/año del accidente, como texto: `12/2025` |
| `gravedad_accidente` | CON HERIDOS o CON MUERTOS |
| `departamento_accidente` | Departamento |
| `municipio_accidente` | Municipio |
| `autoridad_de_transito` | Autoridad que registró el caso |

**Tres advertencias antes de empezar:**

1. Todo el texto está en MAYÚSCULAS y sin tildes. `'Motocicleta'` devuelve 0 filas.
   `'MOTOCICLETA'` funciona.
2. `fecha_accidente` es texto, no fecha. `modelo_vehiculo` es el año del **modelo**, no el del
   accidente.
3. `marca_vehiculo` tiene 4 valores vacíos y `edad_vehiculo` tiene 1. Hoy no se limpian.

## Cómo se recorre este cuaderno

Usted trabaja solo. Nadie va a dictar los pasos desde el tablero, así que cada tarea trae todo lo
que necesita para resolverse leyendo:

| Parte de la tarea | Qué contiene |
|-------------------|--------------|
| **La pregunta** | Lo que hay que responder, escrito en español, como lo pediría alguien de la entidad |
| **El concepto** | Qué técnica aplica y por qué esa y no otra |
| **Los comandos** | Las instrucciones exactas que va a usar, escritas de forma genérica |
| **Lo que decide usted** | Qué columna, qué valor y qué operador. Ahí no hay respuesta escrita |
| **La celda de código** | Los pasos numerados en comentarios. Usted escribe las líneas |
| **La comprobación** | `comprobar('TN', ...)` le dice si el número es el correcto, sin mostrárselo |

**Por qué esto sigue siendo un reto y no una copia.** En el demo trabajó sobre 318 filas de estados
financieros. Aquí hay 20.000 filas de accidentes de tránsito que no ha visto nunca. Le damos el
camino —los comandos, la técnica, el orden—, pero el camino lo recorre usted sobre datos nuevos:
elige la columna, escribe el valor exacto tal como está en el dato, decide el operador, y **dice qué
significa el número que sale**. La técnica se guía; el criterio no se guía, y el criterio es lo que
se evalúa.

**Las celdas `comprobar(...)`** comparan una huella digital de su resultado con la esperada. Si
coinciden, dicen `CORRECTO`; si no, le dan una pista dirigida al error más probable. Nunca le
muestran la respuesta: si escribe cualquier cosa hasta que pase, se está engañando en un cuaderno
que además es su entregable.

**Las celdas `Tu respuesta:`** no llevan código. Son las que se leen en la dimensión Saber. Un
cuaderno con los diez números correctos y ninguna frase escrita está a medias.

**Al final** hay un punto de control que le dice cuántas de las diez tareas quedaron correctas.

---

## Paso 0 · Cargar los datos

**El concepto.** `pd.read_csv()` lee el archivo del disco y lo copia a la memoria como un
**DataFrame** (la tabla de pandas). Los dos puntos del principio de la ruta son la instrucción:
`../` significa "suba un nivel desde la carpeta donde está este cuaderno", y lo que sigue es la
carpeta de datos y el nombre del archivo. La ruta es relativa **al cuaderno**, no a la carpeta que
tenga abierta en VSCode: ese es el error número uno del semestre.

**Los comandos.**

```python
df = pd.read_csv('ruta/al/archivo.csv')
df.shape[0]   # cuantas filas
df.shape[1]   # cuantas columnas
df.head()     # las primeras 5 filas
```

Esta celda ya está escrita. Ejecútela y confirme que dice 20.000 filas y 9 columnas.

In [ ]:
import pandas as pd

df = pd.read_csv('../datos/vehiculos_accidentes.csv')

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
df.head()

In [ ]:
# Verificador de las tareas. Ejecute esta celda una vez y siga adelante.
# No hace falta entenderla hoy: es andamiaje del curso, no materia de la clase.
import hashlib

_RESULTADOS = {}

_PISTAS = {
    "T1": "Compare la columna tipo_vehiculo con el texto exacto que salio en la celda de reconocimiento. Esta todo en mayusculas. La respuesta es un numero de filas, no una tabla.",
    "T2": "modelo_vehiculo ya es numerica: se compara con el numero 2020 sin comillas. '2020 o posterior' incluye el 2020, asi que el operador es >=.",
    "T3": "El valor exacto de la columna gravedad_accidente esta en la celda de reconocimiento. Son dos palabras y van en mayusculas.",
    "T4": "Dos condiciones unidas con &, cada una entre parentesis. Si le sale un TypeError, le faltan parentesis; si le sale ValueError, escribio 'and' en vez de &.",
    "T5": "Es un o, no un y: ninguna fila puede ser de dos departamentos a la vez. El operador es | y cada condicion va entre parentesis.",
    "T6": "Las dos formas tienen que dar el mismo numero. Con ~ la condicion completa va entre parentesis: ~(df['col'] == valor). Guarde el conteo, no el DataFrame.",
    "T7": "isin recibe una lista entre corchetes con los cuatro tipos, escritos igual que en el dato. Revise que no se le quede uno por fuera.",
    "T8": "between(0, 3) incluye los dos extremos. Se aplica sobre edad_vehiculo, que es la antiguedad en anios, no sobre modelo_vehiculo, que es el anio del modelo.",
    "T9": "Cuatro condiciones con & y parentesis en cada una, la de los departamentos con isin. Al final seleccione las cinco columnas pedidas, en el orden pedido, con dos pares de corchetes.",
    "T10": "Cada porcentaje es fatales de ese tipo dividido entre el total de ese tipo, por 100. No divida entre las 20.000 filas: la pregunta es dentro de cada tipo."
}

_ESPERADO = {
    "T1": "72fafba2b6",
    "T2": "328b716208",
    "T3": "ed223736db",
    "T4": "893e613f5b",
    "T5": "352396cede",
    "T6": "6b611b270b",
    "T7": "c62e608595",
    "T8": "d93bae4acb",
    "T9": "479fba96e2",
    "T10": "76cce935f7"
}


def _firma(valor):
    """Reduce un resultado a un texto reproducible, sin importar como se calculo."""
    if isinstance(valor, (list, tuple)):
        return "lista|" + "|".join(_firma(v) for v in valor)
    if isinstance(valor, pd.DataFrame):
        partes = ["DataFrame", str(valor.shape), str([str(c) for c in valor.columns]),
                  str([str(i) for i in valor.index])]
        for columna in valor.columns:
            serie = valor[columna]
            if pd.api.types.is_bool_dtype(serie) or not pd.api.types.is_numeric_dtype(serie):
                partes.append(f"{columna}:{[str(v) for v in serie.tolist()]}")
            else:
                partes.append(f"{columna}:{round(float(serie.sum()), 4)}")
        return "|".join(partes)
    if isinstance(valor, pd.Series):
        return "|".join(["Series", str(len(valor)), str([str(i) for i in valor.index]),
                         str([str(v) for v in valor.tolist()])])
    if not isinstance(valor, str):
        try:
            return f"numero|{round(float(valor), 4)}"
        except (TypeError, ValueError):
            pass
    return f"otro|{valor!r}"


def _huella(valor):
    return hashlib.sha256(_firma(valor).encode("utf-8")).hexdigest()[:10]


def comprobar(clave, valor):
    """Dice si el resultado de la tarea es el correcto, sin revelar cual era."""
    _RESULTADOS[clave] = False
    if valor is None:
        print(f"[{clave}] Sin resolver todavia: la variable sigue valiendo None.")
        return
    if isinstance(valor, pd.DataFrame):
        print(f"[{clave}] Usted produjo un DataFrame de {valor.shape[0]} filas "
              f"y {valor.shape[1]} columnas.")
    elif isinstance(valor, pd.Series):
        print(f"[{clave}] Usted produjo una Series de {len(valor)} elementos, tipo {valor.dtype}.")
    elif isinstance(valor, (list, tuple)):
        print(f"[{clave}] Usted produjo: {[float(v) for v in valor]}")
    else:
        print(f"[{clave}] Usted produjo: {valor!r}")
    if _huella(valor) == _ESPERADO[clave]:
        _RESULTADOS[clave] = True
        print(f"[{clave}] CORRECTO.")
    else:
        print(f"[{clave}] Todavia no coincide.")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")


def resumen_puntos_de_control():
    """Estado de las diez tareas."""
    orden = [f"T{n}" for n in range(1, 11)]
    print("Punto de control final")
    print("-" * 40)
    for clave in orden:
        estado = "correcta" if _RESULTADOS.get(clave) else "pendiente"
        print(f"  {clave}: {estado}")
    logradas = sum(1 for c in orden if _RESULTADOS.get(c))
    print("-" * 40)
    print(f"{logradas} de {len(orden)} tareas correctas.")


print("Verificador listo. Las tareas se comprueban con comprobar('T1', su_variable).")

### Paso 0.1 · Reconocimiento: mire los valores antes de filtrar

**El concepto.** El error más frecuente de esta clase no es de sintaxis: es de supuesto. Usted
escribe `'Motocicleta'`, el dato dice `'MOTOCICLETA'`, el filtro devuelve cero filas y el código se
ve impecable. `.unique()` devuelve los valores que **realmente** existen en una columna, sin
repeticiones. Ejecútelo antes de escribir el primer filtro y téngalo a mano toda la sesión: de aquí
va a copiar y pegar los textos exactos de casi todas las tareas.

**Los comandos.**

```python
df['columna'].unique()          # los valores distintos, como estan escritos
df['columna'].unique().tolist() # los mismos, como lista de Python, mas facil de leer
df['columna'].dropna()          # sin los vacios, para poder ordenarlos
```

Esta celda también está escrita. Ejecútela y **no cierre la salida**.

In [ ]:
print("Tipos de vehículo:", df['tipo_vehiculo'].unique().tolist())
print()
print("Gravedad:", df['gravedad_accidente'].unique().tolist())
print()
print("Departamentos:", sorted(df['departamento_accidente'].dropna().unique().tolist()))

---

## Parte 1 · Una sola condición

**Qué se practica aquí.** El patrón básico de todo el semestre: hacerle **una** pregunta a **una**
columna, y quedarse con las filas que responden que sí.

**El concepto, en dos pasos.** Filtrar nunca es un paso, son dos:

1. **Construir la máscara.** `df['columna'] == 'VALOR'` no filtra nada: devuelve una **Series de
   `True` y `False`**, del mismo largo que el DataFrame, con una marca por fila. Es la planilla
   después de pasar lista: sí, no, no, sí.
2. **Aplicar la máscara.** `df[mascara]` devuelve una tabla nueva, solo con las filas donde la
   máscara dijo `True`. El `df` original no cambia nunca.

**El atajo que va a usar todo el día.** Cuando la pregunta empieza por "¿cuántos...?", la respuesta
es un **número**, no una tabla. Y para contar no hace falta filtrar: `mascara.sum()` cuenta los
`True` directamente, porque en pandas `True` vale 1 y `False` vale 0. `len(df_filtrado)` y
`df_filtrado.shape[0]` dan lo mismo; use el que le resulte más claro.

| Operador | Significado |
|----------|-------------|
| `==` | igual a (dos signos igual: uno solo es asignación) |
| `!=` | distinto de |
| `>` `<` | mayor que, menor que |
| `>=` `<=` | mayor o igual, menor o igual |

### Tarea 1 · Motocicletas

**La pregunta.** ¿Cuántos de los 20.000 vehículos registrados son motocicletas?

**El concepto.** Una sola condición de igualdad sobre una columna de texto. El valor tiene que estar escrito **exactamente** como aparece en el dato: en mayúsculas y sin tildes. Cópielo de la salida de la celda de reconocimiento en vez de escribirlo de memoria.

**Los comandos.**

```python
mascara = df['columna'] == 'VALOR EXACTO'
n = mascara.sum()          # cuantos True
df_filtrado = df[mascara]  # la tabla con esas filas
porcentaje = n / len(df) * 100
```

**Lo que decide usted.** Cuál es la columna, cuál es el texto exacto del valor, y si le sirve más el conteo o la tabla filtrada.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Construya la máscara sobre la columna tipo_vehiculo y guárdela en una variable.
# 2. Guarde en n_motos cuántas filas cumplen (use .sum() sobre la máscara).
# 3. Imprima n_motos y el porcentaje que representa sobre el total.

n_motos = None

In [ ]:
comprobar('T1', n_motos)

**Tu respuesta:** ¿Le sorprende ese porcentaje? ¿Qué dice sobre el parque automotor colombiano o sobre quién sufre los accidentes?

*Tu respuesta:*

### Tarea 2 · Vehículos nuevos

**La pregunta.** ¿Cuántos vehículos tienen un modelo de 2020 o posterior?

**El concepto.** La misma idea, pero sobre una columna **numérica**. `modelo_vehiculo` ya se leyó como entero, así que el valor de comparación va **sin comillas**: `2020`, no `'2020'`. Comparar una columna numérica contra un texto no da error: da una máscara toda en `False`, que es peor, porque parece que no hay datos.

**Los comandos.**

```python
mascara = df['columna'] >= numero   # sin comillas: es un numero
n = mascara.sum()
```

**Lo que decide usted.** El operador. "2020 o posterior" incluye el 2020: eso decide entre `>` y `>=`. Elegir mal aquí no produce ningún error, solo un número equivocado.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Filtre modelo_vehiculo por 2020 o posterior.
# 2. Guarde el conteo en n_nuevos e imprímalo junto con el porcentaje del total.

n_nuevos = None

In [ ]:
comprobar('T2', n_nuevos)

### Tarea 3 · Accidentes fatales

**La pregunta.** ¿Cuántos registros corresponden a accidentes con muertos?

**El concepto.** Igual que la tarea 1, sobre otra columna de texto. Es la última vez que le decimos esto: el valor se copia de `.unique()`, no se escribe de memoria. Si el resultado le da 0 filas, no dude del código: mire otra vez la salida del reconocimiento.

**Los comandos.**

```python
df['columna'].unique()                # para ver como esta escrito el valor
mascara = df['columna'] == 'VALOR EXACTO'
n = mascara.sum()
```

**Lo que decide usted.** El texto exacto del valor y sobre qué columna preguntar.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Filtre gravedad_accidente por el valor que corresponde a accidentes con muertos.
# 2. Guarde el conteo en n_fatales e imprímalo con el porcentaje del total.

n_fatales = None

In [ ]:
comprobar('T3', n_fatales)

**Tu respuesta:** Ese porcentaje, ¿es alto o bajo? ¿Con qué lo compararía para saberlo?

*Tu respuesta:*

---

## Parte 2 · Condiciones combinadas

**Qué se practica aquí.** Hacerle varias preguntas a la tabla y combinar las respuestas.

**El concepto.** Cada condición produce su propia máscara, y las máscaras se combinan entre ellas
elemento por elemento: la decisión 1 de una con la decisión 1 de la otra, la 2 con la 2, hasta el
final.

| Operador | Significado | Cuándo |
|----------|-------------|--------|
| `&` | Y | las dos condiciones tienen que cumplirse a la vez |
| `\|` | O | basta con que una se cumpla |
| `~` | NO | invierte la máscara: los `True` pasan a `False` |

**Las dos reglas que no se negocian:**

1. **Cada condición va entre paréntesis.** Sin excepción. `&` tiene más precedencia que `==` en
   Python, así que sin paréntesis la expresión se agrupa mal y revienta con un `TypeError` o un
   `ValueError`. No es estilo: sin paréntesis el código no corre.
2. **Nunca `and`, `or`, `not`.** Esas tres palabras esperan una sola respuesta sí/no y usted les
   está pasando 20.000. El error es
   `ValueError: The truth value of a Series is ambiguous`, y lo vio en el demo.

**La trampa del idioma.** Cuando alguien dice "quiero los datos de Antioquia **y** del Valle", está
pidiendo un `|`, no un `&`. Ninguna fila puede ser de dos departamentos a la vez, así que un `&` ahí
devuelve cero filas. Traducir del español al operador correcto es la mitad del trabajo de este reto.

### Tarea 4 · Motos fatales

**La pregunta.** ¿Cuántas motocicletas estuvieron involucradas en accidentes con muertos?

**El concepto.** Dos condiciones que se tienen que cumplir **a la vez**: es un `&`. Las dos ya las escribió, en las tareas 1 y 3. Aquí lo único nuevo es juntarlas, y lo único que se rompe son los paréntesis.

**Los comandos.**

```python
mascara = (df['columna_a'] == 'VALOR_A') & (df['columna_b'] == 'VALOR_B')
n = mascara.sum()

# Tambien vale hacerlo en dos variables, y se depura mejor:
m1 = df['columna_a'] == 'VALOR_A'
m2 = df['columna_b'] == 'VALOR_B'
n = (m1 & m2).sum()
```

**Lo que decide usted.** Cuál es el operador correcto (¿y, o?) y dónde van los paréntesis.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Combine la condición de tipo_vehiculo con la de gravedad_accidente usando &.
# 2. Guarde el conteo en n_motos_fatales e imprímalo.
# 3. Imprima también qué porcentaje de TODOS los accidentes fatales son de moto.

n_motos_fatales = None

In [ ]:
comprobar('T4', n_motos_fatales)

### Tarea 5 · Dos departamentos

**La pregunta.** ¿Cuántos registros son de ANTIOQUIA o de VALLE DEL CAUCA?

**El concepto.** Aquí está la trampa del idioma. Uno dice "quiero Antioquia **y** Valle", pero ninguna fila puede tener dos departamentos a la vez: con `&` el resultado es cero. El operador es `|`.

**Los comandos.**

```python
mascara = (df['columna'] == 'VALOR_A') | (df['columna'] == 'VALOR_B')
n = mascara.sum()

# Para ver el desglose por valor:
df[mascara]['columna'].value_counts()
```

**Lo que decide usted.** El operador, contra lo que le dice el español. Y si quiere el total, el desglose, o los dos.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Combine los dos departamentos con |.
# 2. Guarde el conteo total en n_dos_departamentos.
# 3. Imprima además cuántos registros hay de cada departamento por separado.

n_dos_departamentos = None

In [ ]:
comprobar('T5', n_dos_departamentos)

### Tarea 6 · Todo menos motos

**La pregunta.** ¿Cuántos vehículos **no** son motocicletas? Hágalo de **dos formas distintas** (con `!=` y con `~`) y verifique que dan el mismo número.

**El concepto.** Negar una condición se puede escribir de dos maneras. Con una sola condición dan lo mismo y `!=` se lee mejor. `~` se vuelve imprescindible cuando lo que hay que negar es una condición **compuesta**: `~((a) & (b))` es "todo lo que no sea, a la vez, a y b". Escribir eso con `!=` es un ejercicio de lógica innecesario.

**Los comandos.**

```python
mascara_a = df['columna'] != 'VALOR'
mascara_b = ~(df['columna'] == 'VALOR')   # el ~ va delante del parentesis

# Comparar dos resultados:
print(mascara_a.sum() == mascara_b.sum())
```

**Lo que decide usted.** Cómo comprueba que las dos formas coinciden. Comparar los dos números a ojo no es comprobar: hágalo con `==` y que el cuaderno lo imprima.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Forma 1: con !=
# 2. Forma 2: con ~
# 3. Guarde en n_no_motos el conteo (el mismo por las dos formas) e imprima
#    los dos números y si coinciden.

n_no_motos = None

In [ ]:
comprobar('T6', n_no_motos)

---

## Parte 3 · Métodos de conveniencia

**Qué se practica aquí.** Escribir filtros que se puedan leer dentro de seis meses. Todo lo de esta
parte se puede hacer con lo que ya sabe; se usa igual, porque el código se escribe una vez y se lee
muchas.

**Los dos métodos.**

```python
df['columna'].isin(['A', 'B', 'C'])   # equivale a (col == 'A') | (col == 'B') | (col == 'C')
df['columna'].between(a, b)           # equivale a (col >= a) & (col <= b)
```

Los dos devuelven una **máscara**, exactamente igual que `==`. Se aplican igual y se combinan igual
con `&` y `|`.

Dos detalles que se preguntan siempre: `.isin()` recibe **una lista** entre corchetes, y
`.between()` es **inclusivo en los dos extremos** (`between(0, 3)` incluye el 0 y el 3).

### Tarea 7 · Transporte pesado y de pasajeros

**La pregunta.** Cuente los vehículos cuyo tipo esté en la lista `['BUS', 'BUSETA', 'MICROBUS', 'CAMION']`. Escríbalo **primero** con una cadena de `|` y **después** con `.isin()`, y verifique que dan lo mismo.

**El concepto.** Las dos formas son equivalentes y dan el mismo número. La diferencia no es la velocidad —a esta escala no se nota—: es que cuatro condiciones unidas por `|` son cuatro oportunidades de escribir mal un nombre, y una lista es una sola cosa que revisar.

**Los comandos.**

```python
# Forma larga
mascara = ((df['columna'] == 'A') | (df['columna'] == 'B') |
           (df['columna'] == 'C') | (df['columna'] == 'D'))

# Forma corta
mascara = df['columna'].isin(['A', 'B', 'C', 'D'])
```

**Lo que decide usted.** Nada de técnica: la comparación. Escriba las dos y quédese mirándolas antes de responder la pregunta de abajo.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Forma larga: los cuatro tipos unidos por |
# 2. Forma corta: .isin() con la lista
# 3. Guarde el conteo en n_transporte e imprima los dos números y si coinciden.

n_transporte = None

In [ ]:
comprobar('T7', n_transporte)

**Tu respuesta:** Mire las dos versiones. ¿Cuál preferiría encontrarse dentro de seis meses, cuando no se acuerde de nada?

*Tu respuesta:*

### Tarea 8 · Vehículos casi nuevos

**La pregunta.** ¿Cuántos vehículos tenían entre 0 y 3 años de antigüedad (ambos incluidos) al momento del accidente? Use `.between()`.

**El concepto.** Un rango son dos condiciones: mayor o igual que el mínimo, y menor o igual que el máximo. `.between(a, b)` las escribe de una vez y es **inclusivo en los dos extremos**. Ojo con la columna: `edad_vehiculo` es la antigüedad en años; `modelo_vehiculo` es el año del modelo. Son cosas distintas y confundirlas no produce ningún error.

**Los comandos.**

```python
mascara = df['columna'].between(a, b)          # incluye a y b
mascara = (df['columna'] >= a) & (df['columna'] <= b)   # lo mismo, mas largo
n = mascara.sum()
```

**Lo que decide usted.** Cuál de las dos columnas de años responde la pregunta, y qué números son el mínimo y el máximo.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Use .between() sobre la columna de antigüedad.
# 2. Guarde el conteo en n_casi_nuevos e imprímalo con el porcentaje del total.

n_casi_nuevos = None

In [ ]:
comprobar('T8', n_casi_nuevos)

---

## Parte 4 · Preguntas analíticas

**Qué cambia aquí.** Nada de técnica: estas dos tareas no usan un solo comando que no haya usado ya.
Lo que cambia es que **la pregunta viene en español y usted la arma**. Le decimos qué comandos
entran en juego; el orden y el ensamblaje son suyos.

Los comandos disponibles son estos, todos vistos:

```python
df['columna'] == 'VALOR'          df['columna'] <= numero
df['columna'].isin([...])         df['columna'].between(a, b)
mascara_1 & mascara_2             mascara.sum()
df[mascara]                       df[['col1', 'col2']]
df.shape[0]                       len(df)
```

Esta parte se empieza en el salón y se termina en casa. Si se atasca, vuelva a partir el problema:
escriba **una** condición, mírela con `.sum()`, y solo después añada la siguiente.

### Tarea 9 · Motos nuevas y fatales en el Eje Cafetero

**La pregunta.** ¿Cuántas motocicletas de 3 años o menos estuvieron en accidentes **con muertos** en CALDAS, RISARALDA o QUINDIO? Además del conteo, muestre el resultado con las columnas `marca_vehiculo`, `modelo_vehiculo`, `edad_vehiculo`, `departamento_accidente` y `municipio_accidente`, en ese orden.

**El concepto.** Cuatro condiciones a la vez, más una selección de columnas. Ningún comando nuevo: tipo de vehículo, antigüedad, gravedad y una lista de departamentos, todas unidas con `&`. La selección de columnas se hace con **dos pares de corchetes**, porque quiere una tabla y no una columna suelta.

**Los comandos.**

```python
# Los cuatro comandos que necesita, ya usados:
df['columna'] == 'VALOR'
df['columna'] <= numero
df['columna'].isin([...])
df[mascara][['col1', 'col2', 'col3']]   # filtrar y despues elegir columnas
```

**Lo que decide usted.** Cómo arma las cuatro condiciones, cuál de ellas se escribe mejor con `.isin()`, y en qué orden pone las columnas. Guarde el resultado final —filtrado **y** con las cinco columnas, en el orden pedido— en `motos_eje`.

In [ ]:
# TU CÓDIGO AQUÍ
# Condición 1: tipo_vehiculo es MOTOCICLETA
# Condición 2: edad_vehiculo de 3 años o menos
# Condición 3: gravedad_accidente es CON MUERTOS
# Condición 4: departamento_accidente está en el Eje Cafetero (use .isin)
# Combine con & y seleccione las cinco columnas pedidas, en ese orden.
# Guarde el resultado en motos_eje, imprima cuántas filas tiene y muéstrelo.

motos_eje = None

In [ ]:
comprobar('T9', motos_eje)

**Tu respuesta:** Con tan pocos casos, ¿se puede concluir algo? ¿Qué le haría falta para sacar una conclusión seria?

*Tu respuesta:*

### Tarea 10 · ¿Qué es más letal, una moto o un carro?

**La pregunta.** Calcule qué porcentaje de los accidentes de **motocicleta** fueron con muertos, y qué porcentaje de los de **automóvil** fueron con muertos. Compare.

**El concepto.** Es la primera pregunta del reto que no se responde con un filtro, sino con **dos filtros y una división**. No se trata de comparar cuántos muertos hay en cada categoría: hay muchas más motos que automóviles en el dataset, así que el número absoluto engaña. Lo que se compara es la **proporción dentro de cada tipo**: de todas las motos, qué fracción terminó en muerte; de todos los automóviles, qué fracción.

**Los comandos.**

```python
total = (df['tipo_vehiculo'] == 'TIPO').sum()
fatales = ((df['tipo_vehiculo'] == 'TIPO') &
           (df['gravedad_accidente'] == 'CON MUERTOS')).sum()
porcentaje = fatales / total * 100
```

**Lo que decide usted.** Cuál es el denominador. Dividir entre las 20.000 filas responde otra pregunta distinta, y esa es exactamente la equivocación que hay que evitar aquí.

In [ ]:
# TU CÓDIGO AQUÍ
# Para MOTOCICLETA: total del tipo, fatales del tipo, y el porcentaje -> pct_motos
# Para AUTOMOVIL:   lo mismo -> pct_autos
# Imprima los dos porcentajes.

pct_motos = None
pct_autos = None

In [ ]:
# El redondeo se hace aquí, para que el resultado no dependa de cuántos decimales
# haya decidido imprimir usted.
if pct_motos is None or pct_autos is None:
    comprobar('T10', None)
else:
    comprobar('T10', [round(pct_motos, 2), round(pct_autos, 2)])

**Tu respuesta:** Escriba dos frases de conclusión. Cuidado con la trampa: estos datos son de vehículos **involucrados en accidentes reportados**, no de todos los vehículos que circulan. ¿Qué NO se puede concluir con estos datos?

*Tu respuesta:*

---

## Punto de control

Ejecute la celda de abajo para ver cuántas de las diez tareas quedaron correctas.

Si alguna sigue pendiente, no pase de largo: la clase 3 arranca dando por sabido todo esto. Si está
en el salón, levante la mano ahora, que el profesor está aquí para eso.

In [ ]:
resumen_puntos_de_control()

---

## Parte 5 · Reflexión (en casa)

Responda en español, dos o tres frases por pregunta.

**1. ¿Cuál de las 10 tareas le costó más y por qué?**

*Tu respuesta:*

**2. Piense en el dataset que su equipo eligió para el proyecto. Escriba una pregunta que se
respondería con un filtro, y el filtro que la respondería** (no tiene que ejecutarlo, solo
escribirlo).

*Tu respuesta:*

---

## Opcional · Solo si terminó todo

Estas dos no se comprueban ni entran en la retroalimentación. Son un anticipo de la clase 3.

**A. ¿Cuántos accidentes ocurrieron en 2024?** La columna `fecha_accidente` es texto con formato
`12/2024`, así que aquí no sirve `==`: hay que buscar un **fragmento** dentro del texto.

```python
df['columna'].str.contains('TEXTO', na=False)
```

`.str` es la puerta a las operaciones de texto aplicadas a las 20.000 filas de un golpe, y
`na=False` significa "las filas sin valor cuentan como que no". Escríbalo siempre: es decir
explícitamente qué pasa con lo que falta, en vez de dejarlo al criterio de la versión instalada.

**B. ¿Cuáles son las 5 marcas con más accidentes?** Investigue `.value_counts()`, que cuenta cuántas
veces aparece cada valor de una columna y los devuelve ordenados de mayor a menor. `.head(5)` se
queda con los cinco primeros.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

---

## Antes de entregar

1. **Kernel → Restart and Run All.** Si algo revienta, arréglelo. Un cuaderno que no corre de arriba
   a abajo le pone techo a la dimensión Hacer.
2. Verifique que todas las celdas **Tu respuesta:** están escritas. El número no es el análisis.
3. Guarde como `clase02_reto_APELLIDO.ipynb` y súbalo al aula virtual, antes del inicio de la
   clase 3.